In [1]:
import pandas as pd
import numpy as np

from datasets import load_dataset

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

C:\Users\ACER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
# loading the  dataset from hugging face
dataset = load_dataset("jquigl/imdb-genres")

dataset

DatasetDict({
    train: Dataset({
        features: ['movie title - year', 'genre', 'expanded-genres', 'rating', 'description'],
        num_rows: 238256
    })
    validation: Dataset({
        features: ['movie title - year', 'genre', 'expanded-genres', 'rating', 'description'],
        num_rows: 29809
    })
    test: Dataset({
        features: ['movie title - year', 'genre', 'expanded-genres', 'rating', 'description'],
        num_rows: 29756
    })
})

In [5]:
df=dataset['train'].to_pandas()

In [6]:
df.head(5)

,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
2,Povjerenje - 2021,Thriller,Thriller,NaN,"In a small unnamed town, in year 2025, Krsto w..."
3,Gulliver Returns - 2021,Fantasy,"Animation, Adventure, Family",4.4,The legendary Gulliver returns to the Kingdom ...
4,Prithvi Vallabh - 1924,Biography,"Biography, Drama, Romance",NaN,"Seminal silent historical film, the story feat..."


I need to keep just the four columns so 

In [8]:
df = df[["movie title - year", "description", "genre", "rating"]]

Now Cleaning the missing values 

In [9]:
df.isna().sum()

movie title - year        0
description               0
genre                     0
rating                69721
dtype: int64

Here rating got large  missing values

In [10]:
df["rating"] = df["rating"].fillna(df["rating"].median())

Step 1 : Text Preprocessing 

In [12]:
import re 

In [13]:
stop_words = set(stopwords.words("english"))

In [14]:
def remove_stopwords(text):
    new_text = []
    
    for word in text.split():
        if word not in stop_words:
            new_text.append(word)
            
    return " ".join(new_text)

In [15]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = remove_stopwords(text)
    return text

In [16]:
df["clean_description"] = df["description"].apply(clean_text)

In [17]:
df[["description", "clean_description"]].head()

,description,clean_description
0,Flaming Ears is a pop sci-fi lesbian fantasy f...,flaming ears pop scifi lesbian fantasy feature...
1,Six people - three couples - meet at random at...,six people three couples meet random dance res...
2,"In a small unnamed town, in year 2025, Krsto w...",small unnamed town year 2025 krsto works agenc...
3,The legendary Gulliver returns to the Kingdom ...,legendary gulliver returns kingdom lilliput gi...
4,"Seminal silent historical film, the story feat...",seminal silent historical film story features ...


In [19]:
df.loc[10, ["description", "clean_description"]]

description          Set during the fading glory of the Austro-Hung...
clean_description    set fading glory austrohungarian empire film t...
Name: 10, dtype: str

In [21]:
tfidf = TfidfVectorizer(max_features=5000)

tfidf_matrix = tfidf.fit_transform(df["clean_description"])

In [22]:
tfidf_matrix.shape

(238256, 5000)

In [25]:
def recommendation_system(movie_title, top_n=5):

    idx = df[df["movie title - year"] == movie_title].index[0]
    
    sim_scores = cosine_similarity(
        tfidf_matrix[idx], tfidf_matrix
    )[0]
    
    sim_indices = sim_scores.argsort()[::-1][1:top_n+1]
    
    return df.iloc[sim_indices][
        ["movie title - year", "genre", "rating"]
    ]

In [26]:
recommendation_system(df["movie title - year"].iloc[10])

,movie title - year,genre,rating
109626,Colonel Redl - 1985,History,7.4
171940,Il camorrista - 1986,Crime,7.1
90036,Swastika Nation - nan,Biography,6.0
198811,La Malibran - 1944,Biography,6.6
41474,His Name Is Scar - nan,Horror,6.0


In [27]:
val_df=dataset["validation"].to_pandas()

In [28]:
val_df["clean_description"]   = val_df["description"].apply(clean_text)

In [29]:
val_tfidf = tfidf.transform(val_df["clean_description"])

In [30]:
def recommend_validation(val_index, top_n=5):

    sim_scores = cosine_similarity(
        val_tfidf[val_index], tfidf_matrix
    )[0]

    sim_indices = sim_scores.argsort()[::-1][:top_n]

    return df.iloc[sim_indices][
        ["movie title - year", "genre", "rating"]
    ]

In [31]:
recommend_validation(10)

,movie title - year,genre,rating
155356,Gelinin muradi - 1957,Romance,5.6
237077,The Hero in Northeast - 1987,War,6.0
167691,Anja - 2018,Family,6.0
232151,Osvobozhdyonnaya zemlya - 1946,War,6.1
137259,Srirangapuram - 2022,Action,9.3


In [32]:
#Evaluation

In [39]:
def evaluate_similarity(val_index):

    sim_scores = cosine_similarity(
        val_tfidf[val_index], tfidf_matrix
    )[0]

    sim_indices = sim_scores.argsort()[::-1][:5]

    # take similarity scores of top 5
    top_scores = sim_scores[sim_indices]

    return float(top_scores.mean())

In [40]:
evaluate_similarity(10)

0.3143547980924168